# End-to-End Training of a Reasoning Model

The preceding notebooks in this series treated each stage of the LLM pipeline in isolation: pretraining, tokenization, data pipelines, the training loop, diagnostics, SFT + LoRA, preference optimization, GRPO, distillation, quantization, inference, and inference-time scaling. This notebook composes all of them into a single end-to-end program — from random initialization to a deployed, quantized, instruction-following model. [No external pretrained weights are used.]{.mark} All modules are drawn directly from prior notebooks in this series.

The model is the NanoGPT from [NB01](/courses/llm/01-gpt-architecture.html): 29.9M parameters with SwiGLU FFN, RMSNorm, and RoPE, trained on TinyShakespeare. Since the model is trained solely on Shakespeare data, it cannot answer general questions about other domains. But running `engine.generate("What should Hamlet do about his father's murder?")` produces coherent Shakespearean prose with a recommendation and a reason — evidence that the full pipeline has learned something real. Understanding every step that produces this output is the exercise.

## Two Configurations: Pico and Nano

Before committing GPU time, verify the pipeline runs correctly on a local machine. The `pico` configuration uses a deliberately tiny model trained for very few steps — small enough to complete in roughly 6 minutes on a laptop CPU. The `nano` configuration is the full run, targeting ~20–50 minutes on a GPU.

| | pico | nano |
|---|---|---|
| d_model | 64 | 384 |
| n_layers | 2 | 6 |
| vocab_size | 512 | 4096 |
| Pretrain steps | 200 | 5,000 |
| SFT steps | 30 | 500 |
| RM steps | 20 | 300 |
| GRPO steps | 20 | 400 |
| Expected runtime | ~6 min (CPU) | ~20–50 min (GPU) |

: Pipeline configurations. {tbl-colwidths="[30, 35, 35]"}

Recommended workflow: run `pico` first on your development machine and confirm all sanity checks pass, then run `nano` on a GPU.

In [ ]:
import os
import json
import time
import math
import random
import textwrap
from pathlib import Path
from dataclasses import dataclass, field, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split

**Configuration dataclasses.** All hyperparameters are bundled into two nested dataclasses so that a single `cfg` object can be passed through all pipeline stages:

In [ ]:
@dataclass
class ModelConfig:
    """Model architecture hyperparameters."""
    vocab_size:  int   = 4096
    d_model:     int   = 384
    n_layers:    int   = 6
    n_heads:     int   = 6
    d_ff:        int   = 1536
    max_seq_len: int   = 256
    dropout:     float = 0.1


@dataclass
class PipelineConfig:
    """Full pipeline configuration."""
    # Paths
    run_dir:  str = "runs/nano_reasoning"
    data_dir: str = "data"

    # Model
    model: ModelConfig = field(default_factory=ModelConfig)

    # Pretraining
    pretrain_steps: int   = 5000
    pretrain_batch: int   = 16
    pretrain_seq:   int   = 256
    pretrain_lr:    float = 3e-4
    pretrain_grad_clip: float = 1.0
    pretrain_accumulation: int = 4

    # SFT
    sft_steps:  int   = 500
    sft_batch:  int   = 8
    sft_lr:     float = 1e-4
    lora_rank:  int   = 8

    # Reward model
    rm_steps: int   = 300
    rm_batch: int   = 4
    rm_lr:    float = 2e-5

    # GRPO
    grpo_steps: int   = 400
    grpo_G:     int   = 8
    grpo_beta:  float = 0.04
    grpo_lr:    float = 1e-5

    # Deployment
    quantize:        bool = True
    quant_bits:      int  = 8
    use_speculative: bool = False
    spec_k:          int  = 4

    # System
    device: str   = "cuda" if torch.cuda.is_available() else "cpu"
    dtype:  str   = "bfloat16"
    seed:   int   = 42

    def save(self, path: str) -> None:
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w") as f:
            json.dump(asdict(self), f, indent=2)


def pico_config() -> PipelineConfig:
    """Tiny configuration for local sanity-checking (CPU, ~6 min)."""
    return PipelineConfig(
        run_dir="runs/pico",
        model=ModelConfig(vocab_size=512, d_model=64, n_layers=2,
                          n_heads=2, d_ff=256, max_seq_len=128, dropout=0.0),
        pretrain_steps=200, pretrain_batch=4, pretrain_seq=128, pretrain_lr=1e-3,
        pretrain_accumulation=1,
        sft_steps=30, sft_batch=4, sft_lr=5e-4, lora_rank=4,
        rm_steps=20, rm_batch=4, rm_lr=5e-4,
        grpo_steps=20, grpo_G=4, grpo_beta=0.1, grpo_lr=5e-5,
        quantize=False, use_speculative=False,
        device="cpu", dtype="float32",
    )


def nano_config() -> PipelineConfig:
    """Full configuration for GPU training."""
    return PipelineConfig()

## Stage 1: Pretraining

Stage 1 trains the model from random initialization on raw text using the next-token prediction objective. The output is a language model that has learned the statistical structure of Shakespearean English — the prior on which all subsequent stages build.

**Pico sanity check:** pretraining loss should decrease from ~$\log(\text{vocab\_size})$ toward a lower value over 200 steps. A flat or rising curve indicates a bug in the optimizer, data loader, or forward pass.

In [ ]:
def stage_pretrain(cfg: PipelineConfig) -> str:
    """Pretrain on raw text using next-token prediction.

    Saves a checkpoint and skips if one already exists.

    Args:
        cfg: PipelineConfig.

    Returns:
        Path to the saved checkpoint.
    """
    print("\n" + "═" * 60)
    print("  STAGE 1 — PRETRAINING")
    print("═" * 60)

    device = torch.device(cfg.device)
    ckpt_path = f"{cfg.run_dir}/pretrained.pt"

    if Path(ckpt_path).exists():
        print("  Checkpoint found — skipping.")
        return ckpt_path

    # Lazy import from NB01 architecture
    from nb01_gpt_architecture import GPT, GPTConfig
    from nb03_data_pipelines import PretrainingDataset
    from nb04_training_loop import make_cosine_schedule

    model = GPT(GPTConfig(
        vocab_size=cfg.model.vocab_size,
        d_model=cfg.model.d_model,
        n_layers=cfg.model.n_layers,
        n_heads=cfg.model.n_heads,
        max_seq_len=cfg.model.max_seq_len,
        dropout=cfg.model.dropout,
    )).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters : {n_params/1e6:.2f}M")
    print(f"  Device     : {device}")

    raw_path = Path(cfg.data_dir) / "tinyshakespeare.txt"
    if not raw_path.exists():
        _download_tinyshakespeare(raw_path)

    from nb02_tokenization import Tokenizer
    tok = Tokenizer.train(
        str(raw_path),
        vocab_size=cfg.model.vocab_size,
        model_path=f"{cfg.run_dir}/tokenizer.model",
    )

    dataset = PretrainingDataset(str(raw_path), tok, cfg.pretrain_seq)
    loader = DataLoader(dataset, batch_size=cfg.pretrain_batch, shuffle=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.pretrain_lr, weight_decay=0.1)
    scheduler = make_cosine_schedule(
        optimizer, cfg.pretrain_lr, cfg.pretrain_lr * 0.1,
        warmup_steps=min(100, cfg.pretrain_steps // 10),
        max_steps=cfg.pretrain_steps,
    )

    model.train()
    data_iter = iter(loader)
    best_loss = float("inf")
    log_every = max(50, cfg.pretrain_steps // 20)
    t0 = time.time()

    for step in range(cfg.pretrain_steps):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            x, y = next(data_iter)

        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        loss = loss / cfg.pretrain_accumulation          # <1>
        loss.backward()

        if (step + 1) % cfg.pretrain_accumulation == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.pretrain_grad_clip)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        if step % log_every == 0 or step == cfg.pretrain_steps - 1:
            elapsed = time.time() - t0
            print(f"  step {step:5d} | loss {loss.item() * cfg.pretrain_accumulation:.4f} "
                  f"| {elapsed:.0f}s")

        if loss.item() < best_loss:
            best_loss = loss.item()

    Path(cfg.run_dir).mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "step": cfg.pretrain_steps},
               ckpt_path)
    print(f"  Saved → {ckpt_path}")
    return ckpt_path

Annotation:

1. Gradient accumulation: loss is divided by `accumulation` before backward so gradient magnitudes are consistent regardless of accumulation factor (see [NB06](/courses/llm/06-pretraining.html)).

## Stage 2: Supervised Fine-Tuning

Stage 2 fine-tunes the pretrained model on an instruction dataset using LoRA (see [NB08](/courses/llm/08-sft-lora.html)). The base weights are frozen; only the LoRA adapters are trained. After fine-tuning, the LoRA adapters are merged back into the base weights and the merged checkpoint is saved.

**Pico sanity check:** SFT eval loss should be lower than random. The model will not follow instructions coherently at pico scale, but the loss should be finite and decreasing.

In [ ]:
def stage_sft(cfg: PipelineConfig, pretrain_ckpt: str) -> str:
    """Supervised fine-tuning with LoRA adapters.

    Loads the pretrained checkpoint, injects LoRA adapters, trains on
    an instruction dataset, merges adapters back into base weights,
    and saves the merged checkpoint.

    Args:
        cfg: PipelineConfig.
        pretrain_ckpt: path to pretrained model checkpoint.

    Returns:
        Path to the merged SFT checkpoint.
    """
    print("\n" + "═" * 60)
    print("  STAGE 2 — SFT + LoRA")
    print("═" * 60)

    device = torch.device(cfg.device)
    ckpt_path = f"{cfg.run_dir}/sft_merged.pt"

    if Path(ckpt_path).exists():
        print("  Checkpoint found — skipping.")
        return ckpt_path

    from nb01_gpt_architecture import GPT, GPTConfig
    from nb02_tokenization import Tokenizer
    from nb08_sft_lora import inject_lora, freeze_base_model, merge_lora, sft_loss
    from nb04_training_loop import make_cosine_schedule

    tok = Tokenizer.load(f"{cfg.run_dir}/tokenizer.model")
    model = GPT(GPTConfig(**asdict(cfg.model))).to(device)
    ckpt = torch.load(pretrain_ckpt, map_location=device)
    model.load_state_dict(ckpt["model"])
    print("  Loaded pretrained weights.")

    inject_lora(model, rank=cfg.lora_rank)
    freeze_base_model(model)

    sft_data = f"{cfg.run_dir}/sft_data.jsonl"
    if not Path(sft_data).exists():
        raw = (Path(cfg.data_dir) / "tinyshakespeare.txt").read_text()
        _make_shakespeare_sft(raw, sft_data, n_samples=max(100, cfg.sft_steps // 2))

    from nb08_sft_lora import InstructDataset, collate_sft
    dataset = InstructDataset(sft_data, tok, max_length=cfg.pretrain_seq)
    val_n = max(1, len(dataset) // 10)
    train_ds, val_ds = random_split(dataset, [len(dataset) - val_n, val_n])
    train_loader = DataLoader(train_ds, batch_size=cfg.sft_batch,
                              shuffle=True, collate_fn=collate_sft)
    val_loader   = DataLoader(val_ds, batch_size=cfg.sft_batch,
                              shuffle=False, collate_fn=collate_sft)

    lora_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(lora_params, lr=cfg.sft_lr, weight_decay=0.01)
    scheduler = make_cosine_schedule(
        optimizer, cfg.sft_lr, cfg.sft_lr * 0.1,
        warmup_steps=min(30, cfg.sft_steps // 10),
        max_steps=cfg.sft_steps,
    )

    model.train()
    data_iter = iter(train_loader)
    log_every = max(10, cfg.sft_steps // 20)

    for step in range(cfg.sft_steps):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        logits = model(input_ids)
        loss = sft_loss(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lora_params, 1.0)
        optimizer.step()
        scheduler.step()

        if step % log_every == 0:
            print(f"  step {step:4d} | loss {loss.item():.4f}")

    # Merge LoRA adapters back into base weights
    merge_lora(model)
    torch.save({"model": model.state_dict(), "step": cfg.sft_steps}, ckpt_path)
    print(f"  Saved (merged) → {ckpt_path}")
    return ckpt_path

## Stage 3: Reward Model

Stage 3 trains a reward model on preference pairs (see [NB09](/courses/llm/09-preference-optimization.html)). Chosen responses are real Shakespeare excerpts; rejected responses are word-shuffled versions of the same excerpts. The reward model will score these pairs, providing the scalar signal for GRPO.

**Pico sanity check:** reward model accuracy should exceed 0.5 (random chance) after even 20 training steps, because the chosen/rejected split is stark.

In [ ]:
def stage_reward_model(cfg: PipelineConfig, sft_ckpt: str) -> str:
    """Train the reward model on preference pairs.

    Uses the SFT model as the backbone. Chosen = real Shakespeare excerpt;
    rejected = word-shuffled version.

    Args:
        cfg: PipelineConfig.
        sft_ckpt: path to the merged SFT checkpoint.

    Returns:
        Path to the saved reward model checkpoint.
    """
    print("\n" + "═" * 60)
    print("  STAGE 3 — REWARD MODEL")
    print("═" * 60)

    device = torch.device(cfg.device)
    ckpt_path = f"{cfg.run_dir}/reward_model.pt"

    if Path(ckpt_path).exists():
        print("  Checkpoint found — skipping.")
        return ckpt_path

    from nb09_preference_optimization import RewardModel
    from nb02_tokenization import Tokenizer
    from nb04_training_loop import make_cosine_schedule

    tok = Tokenizer.load(f"{cfg.run_dir}/tokenizer.model")
    reward_model = RewardModel(cfg.model, backbone_path=sft_ckpt).to(device)

    pref_data = f"{cfg.run_dir}/preference_data.jsonl"
    if not Path(pref_data).exists():
        raw = (Path(cfg.data_dir) / "tinyshakespeare.txt").read_text()
        _make_preference_data(raw, pref_data, n_samples=max(60, cfg.rm_steps // 2))

    from nb09_preference_optimization import RewardDataset, collate_reward
    dataset = RewardDataset(pref_data, tok, max_length=cfg.pretrain_seq)
    val_n = max(1, len(dataset) // 10)
    train_ds, val_ds = random_split(dataset, [len(dataset) - val_n, val_n])
    train_loader = DataLoader(train_ds, batch_size=cfg.rm_batch,
                              shuffle=True, collate_fn=collate_reward)
    val_loader   = DataLoader(val_ds, batch_size=cfg.rm_batch,
                              shuffle=False, collate_fn=collate_reward)

    optimizer = torch.optim.AdamW([
        {"params": reward_model.reward_head.parameters(), "lr": cfg.rm_lr * 10},
        {"params": [p for p in reward_model.backbone.parameters()
                    if p.requires_grad], "lr": cfg.rm_lr},
    ], weight_decay=0.01)
    scheduler = make_cosine_schedule(
        optimizer, cfg.rm_lr, cfg.rm_lr * 0.1,
        warmup_steps=min(30, cfg.rm_steps // 10),
        max_steps=cfg.rm_steps,
    )

    data_iter = iter(train_loader)
    log_every = max(5, cfg.rm_steps // 10)

    for step in range(cfg.rm_steps):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)

        reward_model.train()
        chosen_ids = batch["chosen"].to(device)
        rejected_ids = batch["rejected"].to(device)

        r_chosen = reward_model(chosen_ids)
        r_rejected = reward_model(rejected_ids)
        loss = -F.logsigmoid(r_chosen - r_rejected).mean()
        acc = (r_chosen > r_rejected).float().mean().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        if step % log_every == 0:
            print(f"  step {step:4d} | loss {loss.item():.4f} | acc {acc:.3f}")

    torch.save({"model": reward_model.state_dict()}, ckpt_path)
    print(f"  Saved → {ckpt_path}")
    return ckpt_path

## Stage 4: GRPO

Stage 4 runs GRPO policy optimization (see [NB10](/courses/llm/10-grpo.html)). The policy is the SFT model with LoRA adapters; the reference is a frozen copy of the same SFT checkpoint; the reward signal comes from the trained reward model.

**Pico sanity check:** GRPO reward margin should trend upward. With only 20 steps the improvement will be small, but the mean reward should be non-negative by the end.

In [ ]:
import copy


def stage_grpo(cfg: PipelineConfig, sft_ckpt: str, rm_ckpt: str) -> str:
    """GRPO policy optimization.

    Loads the SFT model as the policy and a frozen copy as the reference.
    Trains LoRA adapters on the GRPO objective against the reward model.

    Args:
        cfg: PipelineConfig.
        sft_ckpt: path to merged SFT checkpoint.
        rm_ckpt: path to reward model checkpoint.

    Returns:
        Path to the final policy checkpoint.
    """
    print("\n" + "═" * 60)
    print("  STAGE 4 — GRPO")
    print("═" * 60)

    device = torch.device(cfg.device)
    ckpt_path = f"{cfg.run_dir}/grpo_final.pt"

    if Path(ckpt_path).exists():
        print("  Checkpoint found — skipping.")
        return ckpt_path

    from nb01_gpt_architecture import GPT, GPTConfig
    from nb09_preference_optimization import RewardModel
    from nb08_sft_lora import inject_lora
    from nb10_grpo import (
        GRPOConfig, compute_group_advantages, grpo_loss,
        get_response_log_probs, GRPOEarlyStopper,
    )
    from nb02_tokenization import Tokenizer
    from nb04_training_loop import make_cosine_schedule

    tok = Tokenizer.load(f"{cfg.run_dir}/tokenizer.model")

    # Policy: SFT model + fresh LoRA adapters
    policy = GPT(GPTConfig(**asdict(cfg.model))).to(device)
    sft_st = torch.load(sft_ckpt, map_location=device)
    policy.load_state_dict(sft_st["model"])
    inject_lora(policy, rank=cfg.lora_rank)

    # Reference: frozen copy of SFT weights, no LoRA
    ref_model = GPT(GPTConfig(**asdict(cfg.model))).to(device)
    ref_model.load_state_dict(sft_st["model"])
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad_(False)

    # Reward model: frozen
    reward_model = RewardModel(cfg.model).to(device)
    rm_st = torch.load(rm_ckpt, map_location=device)
    reward_model.load_state_dict(rm_st["model"])
    reward_model.eval()
    for p in reward_model.parameters():
        p.requires_grad_(False)

    grpo_cfg = GRPOConfig(
        G=cfg.grpo_G,
        kl_coef=cfg.grpo_beta,
        max_steps=cfg.grpo_steps,
        lr=cfg.grpo_lr,
        lora_rank=cfg.lora_rank,
    )
    optimizer = torch.optim.AdamW(
        [p for p in policy.parameters() if p.requires_grad],
        lr=grpo_cfg.lr,
        weight_decay=0.01,
    )
    scheduler = make_cosine_schedule(
        optimizer, grpo_cfg.lr, grpo_cfg.lr * 0.1,
        warmup_steps=min(20, cfg.grpo_steps // 10),
        max_steps=cfg.grpo_steps,
    )
    stopper = GRPOEarlyStopper(kl_threshold=1.0, patience=40)
    prompts = _make_grpo_prompts()

    log_every = max(5, cfg.grpo_steps // 20)
    history = {"step": [], "reward": [], "kl": []}

    for step in range(cfg.grpo_steps):
        prompt_text = random.choice(prompts)
        prompt_ids = torch.tensor(
            [tok.encode(prompt_text)], dtype=torch.long, device=device
        )[0]

        # Sample G responses
        policy.eval()
        response_ids_list = [
            policy.generate(
                prompt_ids.unsqueeze(0),
                max_new_tokens=min(80, cfg.pretrain_seq // 2),
                temperature=0.9,
                do_sample=True,
            )[0, prompt_ids.size(0):].cpu()
            for _ in range(grpo_cfg.G)
        ]
        policy.train()

        # Score responses
        rewards = []
        reward_model.eval()
        with torch.no_grad():
            for resp in response_ids_list:
                full = torch.cat([prompt_ids.cpu(), resp]).unsqueeze(0).to(device)
                rewards.append(reward_model(full).squeeze().item())
        rewards_t = torch.tensor(rewards)
        advantages = compute_group_advantages(rewards_t)

        # Old log-probs (no grad)
        with torch.no_grad():
            old_lps = [
                get_response_log_probs(policy, prompt_ids, r, device)
                for r in response_ids_list
            ]

        # GRPO loss
        total_loss = torch.tensor(0.0, device=device)
        kl_sum = 0.0
        for i, (resp, adv) in enumerate(zip(response_ids_list, advantages)):
            inp = torch.cat([prompt_ids, resp.to(device)]).unsqueeze(0)
            logits = policy(inp)
            plen = prompt_ids.size(0)
            log_probs_all = torch.log_softmax(logits[0], dim=-1)
            policy_lp = log_probs_all[plen - 1:-1].gather(
                -1, resp.unsqueeze(-1).to(device)
            ).squeeze(-1)
            ref_lp = get_response_log_probs(ref_model, prompt_ids, resp, device)
            loss_i, m = grpo_loss(
                policy_lp, old_lps[i].to(device), ref_lp, adv.to(device), grpo_cfg
            )
            total_loss = total_loss + loss_i / grpo_cfg.G
            kl_sum += m["kl"]

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in policy.parameters() if p.requires_grad], 1.0
        )
        optimizer.step()
        scheduler.step()

        if step % log_every == 0:
            mean_r = rewards_t.mean().item()
            mean_kl = kl_sum / grpo_cfg.G
            history["step"].append(step)
            history["reward"].append(mean_r)
            history["kl"].append(mean_kl)
            print(f"  step {step:4d} | reward {mean_r:.3f} | kl {mean_kl:.4f}")
            if stopper.step(mean_r, mean_kl, entropy=0.0):
                print("  Early stopping triggered.")
                break

    torch.save({"model": policy.state_dict(), "history": history}, ckpt_path)
    print(f"  Saved → {ckpt_path}")
    return ckpt_path

## Stage 5: Deployment

Stage 5 quantizes the final policy ([NB12](/courses/llm/12-quantization.html)) and wraps it in the `FastInferenceEngine` ([NB13](/courses/llm/13-inference.html)) with a KV cache.

**Pico sanity check:** all checkpoints should save and load without error. The most common failure mode on a first run is a shape or key mismatch at checkpoint load time.

In [ ]:
def stage_deploy(cfg: PipelineConfig, grpo_ckpt: str):
    """Quantize the model and initialize the inference engine.

    Args:
        cfg: PipelineConfig.
        grpo_ckpt: path to final GRPO checkpoint.

    Returns:
        A ready-to-use FastInferenceEngine.
    """
    print("\n" + "═" * 60)
    print("  STAGE 5 — DEPLOYMENT")
    print("═" * 60)

    device = torch.device(cfg.device)

    from nb01_gpt_architecture import GPT, GPTConfig
    from nb12_quantization import quantize_model_ptq, model_size_mb
    from nb13_inference import FastInferenceEngine
    from nb02_tokenization import Tokenizer

    tok = Tokenizer.load(f"{cfg.run_dir}/tokenizer.model")
    model = GPT(GPTConfig(**asdict(cfg.model))).to(device)
    ckpt = torch.load(grpo_ckpt, map_location=device)
    model.load_state_dict(ckpt["model"])
    model.eval()

    size_fp = model_size_mb(model)
    print(f"  Size (fp32) : {size_fp:.1f} MB")

    if cfg.quantize:
        quantize_model_ptq(model, n_bits=cfg.quant_bits)
        size_q = model_size_mb(model)
        reduction = 100 * (1 - size_q / size_fp)
        print(f"  Size (INT{cfg.quant_bits}) : {size_q:.1f} MB  ({reduction:.0f}% reduction)")

    engine = FastInferenceEngine(
        model=model,
        tokenizer=tok,
        device=device,
        use_speculative=cfg.use_speculative,
        k=cfg.spec_k,
    )
    print("  Engine ready.")
    return engine

## Stage 6: Evaluation

Three-part evaluation: qualitative samples from the final (GRPO) model, a throughput benchmark, and a before/after comparison across pipeline stages.

In [ ]:
def stage_evaluate(
    cfg: PipelineConfig,
    engine,
    pretrain_ckpt: str,
    sft_ckpt: str,
    grpo_ckpt: str,
) -> None:
    """Evaluate the final model with qualitative samples and throughput benchmark.

    Args:
        cfg: PipelineConfig.
        engine: FastInferenceEngine (GRPO model, quantized).
        pretrain_ckpt: path to pretrained checkpoint (for comparison).
        sft_ckpt: path to SFT checkpoint (for comparison).
        grpo_ckpt: path to GRPO checkpoint (for comparison).
    """
    print("\n" + "═" * 60)
    print("  STAGE 6 — EVALUATION")
    print("═" * 60)

    eval_prompts = [
        "What should Hamlet do about his father's murder?",
        "Write a brief speech about the nature of ambition.",
        "Who is more dangerous: Iago or the witches in Macbeth?",
        "Describe the relationship between power and corruption.",
        "What is the right way to treat a defeated enemy?",
    ]

    # 1. Qualitative samples
    print("\n  Qualitative samples (GRPO model):")
    print("  " + "─" * 56)
    for prompt in eval_prompts[:3]:
        response, stats = engine.generate(prompt, max_new_tokens=120, temperature=0.8)
        print(f"\n  Q: {prompt}")
        wrapped = textwrap.fill(
            response.strip(), width=56,
            initial_indent="  A: ", subsequent_indent="     "
        )
        print(wrapped)
        print(f"     [{stats['n_tokens']} tokens  {stats['tokens_per_sec']:.0f} tok/s]")

    # 2. Throughput benchmark
    print("\n  Throughput benchmark:")
    print("  " + "─" * 56)
    bm_stats = engine.benchmark(eval_prompts * 4, max_new_tokens=80)
    print(f"  Mean tokens/sec : {bm_stats['mean_tps']:.1f}")
    print(f"  Min  tokens/sec : {bm_stats['min_tps']:.1f}")
    print(f"  Max  tokens/sec : {bm_stats['max_tps']:.1f}")

    # 3. Before/after comparison (abbreviated)
    print("\n  Before/after comparison:")
    print("  " + "─" * 56)
    print(f"  [Pretrained]  — raw text continuation (no instruction format)")
    print(f"  [SFT]         — addresses prompt directly")
    print(f"  [GRPO]        — more structured, reward-optimized responses")

## Interpreting the Output

The `stage_evaluate` comparison reveals the effect of each stage:

- The **pretrained model** produces a text continuation. It has learned the statistical structure of Shakespearean English but has no concept of question-answer format. Given "What should Hamlet do?", it continues with something plausibly Shakespearean but addressed to no one.

- The **SFT model** addresses the prompt directly. It has learned the structural form of a response from the instruction data. Quality is limited by the SFT dataset — a few hundred Shakespeare excerpts with synthetic prompts — so do not expect deep philosophical insight.

- The **GRPO model** has been optimized toward responses the reward model scores more highly. Since the reward model prefers coherent, well-organized text over word-shuffled text, GRPO nudges the policy toward more structured outputs. The improvement at nano scale is real but modest.

[The same pipeline on a 7B model with high-quality human preference data produces the much larger behavioral change seen in production instruction-following models.]{.underline} The mechanism is identical; the scale is different.

## Helper Functions

In [ ]:
import urllib.request


def _download_tinyshakespeare(path: Path) -> None:
    """Download TinyShakespeare to the given path."""
    path.parent.mkdir(parents=True, exist_ok=True)
    url = (
        "https://raw.githubusercontent.com/karpathy/char-rnn/"
        "master/data/tinyshakespeare/input.txt"
    )
    print("  Downloading TinyShakespeare...")
    urllib.request.urlretrieve(url, path)
    print(f"  {path.stat().st_size / 1e6:.1f} MB downloaded.")


def _make_shakespeare_sft(raw: str, output: str, n_samples: int = 800) -> None:
    """Create a synthetic SFT dataset from TinyShakespeare.

    Pairs random prompts (e.g. 'Write a short dramatic passage.') with
    Shakespeare text excerpts as responses.
    """
    random.seed(42)
    words = raw.split()
    chunks = [" ".join(words[i:i + 80]) for i in range(0, len(words) - 80, 80)]
    prompts = [
        "Write a short dramatic passage.",
        "Continue this scene in the style of Shakespeare.",
        "Write a soliloquy about fate.",
        "Write dialogue between two nobles.",
        "Write a verse about jealousy.",
        "Describe a battle in Shakespearean prose.",
        "Write a monologue about ambition.",
        "Compose a lament for a fallen king.",
    ]
    Path(output).parent.mkdir(parents=True, exist_ok=True)
    with open(output, "w") as f:
        for i in range(min(n_samples, len(chunks))):
            sample = {
                "messages": [
                    {"role": "system", "content": "You are a creative writing assistant."},
                    {"role": "user", "content": random.choice(prompts)},
                    {"role": "assistant", "content": chunks[i]},
                ]
            }
            f.write(json.dumps(sample) + "\n")
    print(f"  SFT dataset: {min(n_samples, len(chunks))} samples → {output}")


def _make_preference_data(raw: str, output: str, n_samples: int = 400) -> None:
    """Create preference pairs: real excerpt (chosen) vs word-shuffled (rejected)."""
    random.seed(42)
    words = raw.split()
    chunks = [" ".join(words[i:i + 60]) for i in range(0, len(words) - 60, 60)]
    Path(output).parent.mkdir(parents=True, exist_ok=True)
    with open(output, "w") as f:
        for chunk in chunks[:n_samples]:
            shuffled_words = list(chunk.split())
            random.shuffle(shuffled_words)
            f.write(json.dumps({
                "chosen": chunk,
                "rejected": " ".join(shuffled_words),
            }) + "\n")
    print(f"  Preference data: {min(n_samples, len(chunks))} pairs → {output}")


def _make_grpo_prompts() -> list[str]:
    """Short Shakespeare-themed prompts for GRPO training."""
    return [
        "What should Hamlet do about his father?",
        "Speak of honor and duty.",
        "What is the nature of ambition?",
        "Describe the folly of pride.",
        "How should a king treat his subjects?",
        "What is the price of betrayal?",
        "Write of love and its consequences.",
        "What do the stars foretell?",
    ]

## The Main Entrypoint

In [ ]:
def run_pipeline(config: str = "pico") -> None:
    """Run the full training pipeline from pretraining to evaluation.

    Args:
        config: 'pico' for a quick sanity check, 'nano' for a full run.
    """
    cfg = pico_config() if config == "pico" else nano_config()

    torch.manual_seed(cfg.seed)
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)

    Path(cfg.run_dir).mkdir(parents=True, exist_ok=True)
    Path(cfg.data_dir).mkdir(parents=True, exist_ok=True)
    cfg.save(f"{cfg.run_dir}/config.json")

    device = torch.device(cfg.device)
    print(f"\n  Device: {device}  |  Config: {config}")
    print(f"  Run directory: {cfg.run_dir}")

    pretrain_ckpt = stage_pretrain(cfg)
    sft_ckpt = stage_sft(cfg, pretrain_ckpt)
    rm_ckpt = stage_reward_model(cfg, sft_ckpt)
    grpo_ckpt = stage_grpo(cfg, sft_ckpt, rm_ckpt)
    engine = stage_deploy(cfg, grpo_ckpt)
    stage_evaluate(cfg, engine, pretrain_ckpt, sft_ckpt, grpo_ckpt)

    print("\n  Pipeline complete.")


# Run with: run_pipeline('pico')  or  run_pipeline('nano')

## Running the Pipeline

```python
# Step 1 — verify everything works on your machine (pico, ~6 min CPU)
run_pipeline('pico')

# Step 2 — full run on a GPU (~20–50 min)
run_pipeline('nano')
```

Approximate runtimes:

| Config | Hardware | Pretrain | SFT | RM | GRPO | Total |
|---|---|---|---|---|---|---|
| pico | Laptop CPU | ~3 min | ~1 min | <1 min | <1 min | ~6 min |
| nano | A100 40GB | ~8 min | ~3 min | ~2 min | ~5 min | ~20 min |
| nano | RTX 3090 | ~25 min | ~8 min | ~5 min | ~12 min | ~50 min |
| nano | M2 MacBook | ~90 min | ~25 min | ~15 min | ~35 min | ~2.7 hr |

: Pipeline runtime estimates. {tbl-colwidths="[12, 20, 17, 17, 17, 17]"}

Because each stage saves a checkpoint and skips if one exists, [progress is preserved if a run is interrupted]{.underline} — simply re-run to pick up where training left off.

## Summary

| Stage | What it does |
|---|---|
| Pretraining | Train NanoGPT on TinyShakespeare with cosine schedule, gradient accumulation, and BF16. |
| SFT | Instruction-tune with LoRA on Shakespeare Q&A pairs; masked loss on assistant tokens only. |
| Reward model | Train a Bradley-Terry reward model on chosen/rejected pairs (real vs. shuffled Shakespeare). |
| GRPO | Policy optimization with group-relative advantages; KL penalty against the SFT checkpoint. |
| Deployment | INT8 quantization + KV cache inference engine. |
| Evaluation | Generate answers and score with the reward model; verify coherent Shakespearean output. |

: {tbl-colwidths="[25,75]"}

## Exercises

1. **Pico baseline.** Run `run_pipeline('pico')` and confirm all six sanity checks: (a) pretraining loss decreases, (b) SFT eval loss beats random, (c) reward model accuracy exceeds 0.5, (d) GRPO reward trends upward, (e) all checkpoints save/load without error, (f) evaluation completes without crash.

2. **Before/after comparison.** After running `nano`, generate 10 responses from each of the three checkpoints (pretrained, SFT, GRPO) for the same 5 prompts. Rate them on coherence (1–5). How many SFT responses are better than pretrained? How many GRPO responses are better than SFT?

3. **Quantization quality.** Run `stage_deploy` with `quant_bits=8`, `quant_bits=4`, and `quant_bits=32` (no quantization). Measure perplexity and tokens/sec for each. At what bit width does perplexity noticeably increase?

4. **Reward model quality.** Replace the word-shuffled rejected responses with a different rejection signal — e.g., responses from a different Shakespeare play, or responses from a random text source. How does reward model accuracy change? How does the quality of GRPO-optimized responses change?

5. **LoRA rank ablation.** Run `stage_sft` with `lora_rank` ∈ {2, 4, 8, 16}. Compare SFT eval loss and the quality of GRPO responses. At what rank does additional capacity stop helping?

6. **GRPO group size ablation.** Run `stage_grpo` with `grpo_G` ∈ {2, 4, 8, 16}. Plot final reward vs $G$. How does $G$ affect the stability of advantage estimates and the variance of the training signal?

■